# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset with the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their @ids
print("Available Record Sets (by @id):")
record_set_ids = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
for rid in record_set_ids:
    print(f"- {rid}")

# If no record sets found at the root, attempt to extract from the metadata via dataset._record_sets
if not record_set_ids and hasattr(dataset, '_record_sets'):
    record_set_ids = [r['@id'] for r in dataset._record_sets]
    for rid in record_set_ids:
        print(f"- {rid}")

# If still empty, enumerate record sets using the dataset.record_sets attribute
if not record_set_ids:
    try:
        record_sets = list(dataset.record_sets)
        record_set_ids = [r['@id'] for r in record_sets]
        for rs in record_sets:
            print(f"- {rs['@id']} : {rs.get('name','')}")
    except Exception as e:
        print("Could not enumerate record sets:", e)

# For demonstration, attempt to inspect the first record set's fields and columns by @id (if available)
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nFields in record set '{example_record_set_id}':")
    # Try to extract fields by record set @id (this step depends on metadata structure)
    try:
        record_sets_json = metadata.to_json().get('recordSet', [])
        record_set_entry = None
        for rs in record_sets_json:
            if rs['@id'] == example_record_set_id:
                record_set_entry = rs
                break
        if record_set_entry and 'field' in record_set_entry:
            field_ids = [f['@id'] if isinstance(f, dict) else f for f in record_set_entry['field']]
            for fid in field_ids:
                print(f"- {fid}")
        else:
            print("Could not locate fields in this record set via metadata.")
    except Exception as e:
        print(f"Error extracting fields: {e}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we will attempt to guess main record set(s) if available.
# Based on Croissant best practices, we can also use dataset.record_sets or metadata.recordSet to fetch them
if record_set_ids:
    record_sets_to_load = record_set_ids
else:
    # Try fallback, will attempt default Croissant Main record set
    try:
        record_sets = list(dataset.record_sets)
        record_sets_to_load = [rs['@id'] for rs in record_sets]
    except Exception as e:
        print("Could not determine record sets for extraction:", e)
        record_sets_to_load = []

dataframes = {}
for record_set_id in record_sets_to_load:
    print(f"\nExtracting records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Fields (by @id):")
        print(df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Choose main record set for further exploration
main_record_set_id = None
for rid, df in dataframes.items():
    if df.shape[0] > 0:
        main_record_set_id = rid
        break
if not main_record_set_id:
    # Pick first as fallback
    if len(dataframes):
        main_record_set_id = list(dataframes.keys())[0]

# Show columns of chosen record set
if main_record_set_id:
    print(f"\nMain record set chosen: {main_record_set_id}")
    print('Columns:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data extracted from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
import numpy as np

# If there's no main record set, EDA is not possible
df = dataframes.get(main_record_set_id)
if df is not None and len(df):
    # Try to find numeric fields: we'll heuristically search for typical numeric-looking column names
    numeric_field_candidates = [c for c in df.columns if df[c].dtype in [np.int64, np.float64, int, float] or 'age' in c.lower() or 'interval' in c.lower() or 'num' in c.lower()]
    if numeric_field_candidates:
        # Pick the most likely field
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
        # Filter out non-numeric or missing
        df_filtered = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
        df_filtered[numeric_field_id] = pd.to_numeric(df_filtered[numeric_field_id], errors='coerce')
        threshold = np.percentile(df_filtered[numeric_field_id], 60)
        filtered_df = df_filtered[df_filtered[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[numeric_field_id + '_normalized'] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Attempt to group by a likely categorical field
        group_field_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
        group_field = None
        for c in group_field_candidates:
            nunique = df[c].nunique()
            if nunique > 1 and nunique < len(df) / 2:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and len(df) and numeric_field_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=12)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot with a likely categorical field, if it exists and is not too granular
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to programmatically explore the FAIR^2 dataset, loading data directly from its Croissant schema. We reviewed available record sets by their `@id`, inspected the fields, and demonstrated filtering and normalization on a numeric field. Exploratory plots visualized value distributions and relationships between demographic or clinical fields. 

This process can be adapted to any dataset adhering to the Croissant standard: simply switch the Croissant schema URL and use the discovered `@id` values for record sets and fields to tailor analysis to your use case.